# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

1. Ответ: [1. 2. 3.] [0 1 1]
Знак / это обычное деление, и числа становятся дробными.
// это знак деление с отбрасыванием дробной части: 2//4=0, 7//7=1, 6//4=1.

2. Ответ: [4. 4. 4.]
массив из единиц имеет 3 строки и 4 столбца. axis=1 значит складывать вдоль строк, то есть для каждой строки получается сумма 4 единиц. Строк три, поэтому ответов тоже три. Числа с точкой, потому что ones создает дробные числа.

3. Ответ:

[[0 1 2]
 [1 2 3]
 [2 3 4]]

первый массив это строка [0 1 2], второй это столбец [[0],[1],[2]]. NumPy растягивает оба до таблицы 3 на 3 и складывает: в клетке на строке i и столбце j стоит i + j.

4. Ответ: 6.0
корни из 1, 4 и 9 равны 1, 2 и 3, а их сумма 6. Получается дробное число, потому что sqrt всегда возвращает дробные.

5. Ответ: 7.5
arange(1, 5) это 1, 2, 3, 4, а после ** 2 получаются 1, 4, 9, 16. Сумма 30, среднее 30 / 4 = 7.5.

Ваши ответы и объяснения:

1. 
2. 
3. 
4. 
5. 

In [10]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


In [12]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [16]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [20]:
# 1
oid = np.array([r["order_id"] for r in rows])
month = np.array([r["order_date"][:7] for r in rows])

_, idx = np.unique(oid, return_index=True)
clean = np.zeros(len(rows), dtype=bool)
clean[idx] = True
clean = clean & (cust != "TEST")

print(clean.sum())

sep = amount[clean & paid & (month == "2026-09")].sum()
octo = amount[clean & paid & (month == "2026-10")].sum()
print(sep, octo)
print((octo - sep) / sep * 100)


5280
35243410.0 34419520.0
-2.337713632137185


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [18]:
# 2

cat = np.array([r["category"] for r in rows])

cats = np.unique(cat)
months = np.array(["2026-09", "2026-10"])

table = np.zeros((5, 2))
for i in range(5):
    for j in range(2):
        mask = clean & paid & (cat == cats[i]) & (month == months[j])
        table[i, j] = amount[mask].sum()

print(cats)
print(table)

total = table.sum(axis=1)
print(total)
print(cats[total.argmax()])

change = (table[:, 1] - table[:, 0]) / table[:, 0] * 100
print(np.round(change, 1))
print(cats[change.argmin()])

['Дом' 'Книги' 'Красота' 'Одежда' 'Электроника']
[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
[10123520.  2206210.  5289940. 17672830. 34370430.]
Электроника
[ 2.6  7.1  2.6 -7.2 -2.5]
Одежда


сильнее всех упала одежда, на 7%

## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [25]:
# 3

counts = np.zeros((5, 2))
for i in range(5):
    for j in range(2):
        mask = clean & paid & (cat == cats[i]) & (month == months[j])
        counts[i, j] = mask.sum()

print(counts)

avg = table / counts
print(np.round(avg))

print(cats[avg[:, 1].argmax()], round(avg[:, 1].max()))

[[476. 476.]
 [265. 299.]
 [422. 435.]
 [678. 652.]
 [531. 505.]]
[[10499. 10768.]
 [ 4020.  3816.]
 [ 6187.  6159.]
 [13521. 13046.]
 [32773. 33600.]]
Электроника 33600


Самый высокий средний чек в октябре у электроники.

## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [27]:
# 4

cities = np.unique(city)

table = np.zeros((6, 2))
for i in range(6):
    for j in range(2):
        mask = clean & paid & (city == cities[i]) & (month == months[j])
        table[i, j] = amount[mask].sum()

share = table / table.sum(axis=0)
print(cities)
print(np.round(share * 100, 1))
print(share.sum(axis=0))

k = list(cities).index("Шымкент")
print(share[k] * 100)

['Актобе' 'Алматы' 'Астана' 'Атырау' 'Караганда' 'Шымкент']
[[ 7.3  7.4]
 [41.1 42.5]
 [25.6 25.7]
 [ 2.6  2.8]
 [10.4  9.8]
 [12.9 11.7]]
[1. 1.]
[12.93475858 11.74917605]


доля Шымкента упала с 12.9% до 11.7%

## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [29]:
# 5

change = table[:, 1] - table[:, 0]
total_change = change.sum()
print(change)
print(total_change)

contribution = change / total_change * 100
print(cities)
print(np.round(contribution, 1))

[ -36540.  141950. -183690.   45870. -276840. -514640.]
-823890.0
['Актобе' 'Алматы' 'Астана' 'Атырау' 'Караганда' 'Шымкент']
[  4.4 -17.2  22.3  -5.6  33.6  62.5]


Шымкент дает 62.5% общего падения.

## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [ ]:
# 6

## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7

## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8